# Evaluación del Modelo - BLEU Score y Análisis

**Fase 4 (Evaluation) + Fase 6 (Monitoring)**

Evaluación completa del modelo entrenado.
Cálculo de BLEU Score, análisis de errores y ejemplos de traducción.

In [1]:
import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '/home/honorio/IA/rnn/src')

from metrics import BLEUScore, ErrorAnalyzer, exact_match_accuracy
from preprocessing import ParallelDataProcessor
from inference import TranslationInference

base_path = Path('/home/honorio/IA/rnn')
splits_path = base_path / 'data' / 'splits'
processed_path = base_path / 'data' / 'processed'
models_path = base_path / 'models'

print("✓ Setup completado")

I0000 00:00:1777484234.829659   55564 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777484234.829965   55564 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1777484234.859767   55564 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777484235.622458   55564 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

✓ Setup completado


## 1. Cargar Datos y Modelo

In [8]:
# Cargar datos de prueba
X_test = np.load(splits_path / 'X_test.npy')
y_test = np.load(splits_path / 'y_test.npy')

# Cargar vocabularios
with open(processed_path / 'vocab_spanish.json', 'r') as f:
    vocab_spanish = json.load(f)
with open(processed_path / 'vocab_english.json', 'r') as f:
    vocab_english = json.load(f)

# Cargar SavedModel directamente (compatible con Keras 3)
loaded_model = tf.saved_model.load(str(models_path / 'traductor_v1'))
print(f"✓ Modelo cargado")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")
print(f"Vocabularios: ES={len(vocab_spanish)}, EN={len(vocab_english)}")

✓ Modelo cargado
X_test: (10, 30), y_test: (10, 30)
Vocabularios: ES=40, EN=42


## 2. Calcular BLEU Score

In [11]:
# Importar pandas para el timestamp
import pandas as pd

# Realizar predicciones
idx2spanish = {v: k for k, v in vocab_spanish.items()}
idx2english = {v: k for k, v in vocab_english.items()}

bleu_calculator = BLEUScore()

# Generar predicciones usando SavedModel cargado
# Crear decoder_input dummy (mismo tamaño que y_test pero desplazado)
decoder_input_test = np.zeros((X_test.shape[0], y_test.shape[1] - 1), dtype=np.float32)
X_test_float = X_test.astype(np.float32)

# Llamar el modelo SavedModel a través de serving_default con argumentos nombrados
print("Generando predicciones...")
infer = loaded_model.signatures['serving_default']
output = infer(args_0=X_test_float, args_0_1=decoder_input_test)

# Extraer predicciones del output
predictions = output['output_0'].numpy()
predicted_indices = np.argmax(predictions, axis=-1)

# Decodificar predicciones y referencias
predicted_sentences = []
reference_sentences = []

for i in range(min(100, len(y_test))):  # Evaluar primeras 100
    pred_tokens = [idx2english.get(int(idx), '<UNK>') for idx in predicted_indices[i]]
    pred_tokens = [t for t in pred_tokens if t not in ['<PAD>', '<START>', '<END>', '<UNK>']]
    predicted_sentences.append(pred_tokens)
    
    ref_tokens = [idx2english.get(int(idx), '<UNK>') for idx in y_test[i]]
    ref_tokens = [t for t in ref_tokens if t not in ['<PAD>', '<START>', '<END>', '<UNK>']]
    reference_sentences.append(ref_tokens)

# Calcular BLEU
bleu = bleu_calculator.compute_corpus_bleu(predicted_sentences, reference_sentences)
exact_match = exact_match_accuracy(predicted_sentences, reference_sentences)

print(f"📊 RESULTADOS EVALUACIÓN")
print(f"  BLEU Score: {bleu:.4f}")
print(f"  Exact Match: {exact_match:.4f}")

# Guardar resultados
eval_results = {
    'bleu_score': float(bleu),
    'exact_match': float(exact_match),
    'samples_evaluated': len(predicted_sentences),
    'timestamp': str(pd.Timestamp.now())
}

with open(models_path / 'evaluation_metrics.json', 'w') as f:
    json.dump(eval_results, f, indent=2)

print(f"✓ Resultados guardados en models/evaluation_metrics.json")

Generando predicciones...
📊 RESULTADOS EVALUACIÓN
  BLEU Score: 0.0000
  Exact Match: 0.0000
✓ Resultados guardados en models/evaluation_metrics.json


E0000 00:00:1777484368.851360   55564 util.cc:131] oneDNN supports DT_BOOL only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


## 3. Ejemplos de Traducción

In [12]:
print("📝 EJEMPLOS DE TRADUCCIÓN")
print("=" * 80)

for i in range(min(10, len(predicted_sentences))):
    pred = ' '.join(predicted_sentences[i])
    ref = ' '.join(reference_sentences[i])
    
    print(f"\nEjemplo {i+1}:")
    print(f"  Predicción: {pred}")
    print(f"  Referencia: {ref}")

print("\n✓ Evaluación completada")

📝 EJEMPLOS DE TRADUCCIÓN

Ejemplo 1:
  Predicción: i you
  Referencia: my name is juan

Ejemplo 2:
  Predicción: i morning
  Referencia: you are very kind

Ejemplo 3:
  Predicción: i you
  Referencia: where are you from

Ejemplo 4:
  Predicción: i you
  Referencia: it is a nice day

Ejemplo 5:
  Predicción: i
  Referencia: you are welcome

Ejemplo 6:
  Predicción: i morning
  Referencia: i do not understand

Ejemplo 7:
  Predicción: i
  Referencia: you are welcome

Ejemplo 8:
  Predicción: i you
  Referencia: i am from spain

Ejemplo 9:
  Predicción: please
  Referencia: see you later

Ejemplo 10:
  Predicción: i morning
  Referencia: i do not like

✓ Evaluación completada
